# ECDet

In [ ]:
import torch
import torch.nn as nn
from huggingface_hub import PyTorchModelHubMixin

from ecdetseg.engine.edgecrafter.ecvit import ViTAdapter
from ecdetseg.engine.edgecrafter.hybrid_encoder import HybridEncoder
from ecdetseg.engine.edgecrafter.decoder import ECTransformer
from ecdetseg.engine.edgecrafter.postprocessor import PostProcessor


class ECDet(nn.Module, PyTorchModelHubMixin):
    def __init__(self, config):
        super().__init__()
        self.config = config

        self.backbone = ViTAdapter(
            **config["backbone"]
        )
        self.encoder = HybridEncoder(
            **config["encoder"]
        )
        self.decoder = ECTransformer(
            **config["decoder"]
        )
        self.postprocessor = PostProcessor(
            **config["postprocessor"]
        )
    
    def forward(self, x, orig_target_sizes):
        x = self.backbone(x)
        x = self.encoder(x)
        x = self.decoder(x)
        x = self.postprocessor(x, orig_target_sizes)

        return x

In [ ]:
ecdet_s_config = {
    "backbone": {
        "name": "ecvitt",
        "skip_load_backbone": True,
        "interaction_indexes": [10, 11],
        "embed_dim": 192,
        "num_heads": 3,
        "patch_size": 16,
        "proj_dim": None,
        "num_levels": 3,
        "embed_layer": "ConvPyramidPatchEmbed",
        "ffn_layer": "mlp",
        "ffn_ratio": 4,
        "skip_load_backbone": False,
    },
    "encoder": {
        "in_channels": [192, 192, 192],
        "feat_strides": [8, 16, 32],
        "hidden_dim": 192,
        "nhead": 8,
        "dim_feedforward": 512,
        "dropout": 0.0,
        "use_encoder_idx": [2],
        "num_encoder_layers": 1,
        "pe_temperature": 10000,
        "expansion": 0.34,
        "depth_mult": 0.67,
        "act": "silu",
        "eval_spatial_size": None,
        "csp_type": "csp2",
        "fuse_op": "sum",
    },
    "decoder": {
        "num_classes": 80,
        "hidden_dim": 192,
        "num_queries": 300,
        "feat_channels": [192, 192, 192],
        "feat_strides": [8, 16, 32],
        "num_levels": 3,
        "num_points": [3, 6, 3],
        "nhead": 8,
        "num_layers": 4,
        "dim_feedforward": 512,
        "dropout": 0.0,
        "activation": "silu",
        "num_denoising": 100,
        "label_noise_ratio": 0.5,
        "box_noise_scale": 1.0,
        "learn_query_content": False,
        "eval_spatial_size": [640, 640],
        "eval_idx": -1,
        "eps": 1e-2,
        "aux_loss": True,
        "cross_attn_method": "default",
        "query_select_method": "default",
        "reg_max": 32,
        "reg_scale": 4,
        "layer_scale": 1,
        "share_bbox_head": False,
        "share_score_head": False,
        "mask_downsample_ratio": None,
    },
    "postprocessor": {
        "num_classes": 80,
        "use_focal_loss": True,
        "num_top_queries": 300,
        "remap_mscoco_category": False,
    },
}

ecdet_s = ECDet(ecdet_s_config).from_pretrained("Intellindust/ECDet_S")

In [ ]:
ecdet_m_config = {
    "backbone": {
        "name": "ecvittplus",
        "skip_load_backbone": True,
        "interaction_indexes": [10, 11],
        "embed_dim": 256,
        "num_heads": 4,
        "patch_size": 16,
        "proj_dim": None,
        "num_levels": 3,
        "embed_layer": "ConvPyramidPatchEmbed",
        "ffn_layer": "mlp",
        "ffn_ratio": 4,
        "skip_load_backbone": False,
    },
    "encoder": {
        "in_channels": [256, 256, 256],
        "feat_strides": [8, 16, 32],
        "hidden_dim": 256,
        "nhead": 8,
        "dim_feedforward": 512,
        "dropout": 0.0,
        "use_encoder_idx": [2],
        "num_encoder_layers": 1,
        "pe_temperature": 10000,
        "expansion": 0.75,
        "depth_mult": 0.67,
        "act": "silu",
        "eval_spatial_size": None,
        "csp_type": "csp2",
        "fuse_op": "sum",
    },
    "decoder": {
        "num_classes": 80,
        "hidden_dim": 256,
        "num_queries": 300,
        "feat_channels": [256, 256, 256],
        "feat_strides": [8, 16, 32],
        "num_levels": 3,
        "num_points": [3, 6, 3],
        "nhead": 8,
        "num_layers": 4,
        "dim_feedforward": 1024,
        "dropout": 0.0,
        "activation": "silu",
        "num_denoising": 100,
        "label_noise_ratio": 0.5,
        "box_noise_scale": 1.0,
        "learn_query_content": False,
        "eval_spatial_size": [640, 640],
        "eval_idx": -1,
        "eps": 1e-2,
        "aux_loss": True,
        "cross_attn_method": "default",
        "query_select_method": "default",
        "reg_max": 32,
        "reg_scale": 4,
        "layer_scale": 1,
        "share_bbox_head": False,
        "share_score_head": False,
        "mask_downsample_ratio": None,
    },
    "postprocessor": {
        "num_classes": 80,
        "use_focal_loss": True,
        "num_top_queries": 300,
        "remap_mscoco_category": False,
    },
}

ecdet_m = ECDet(ecdet_m_config).from_pretrained("Intellindust/ECDet_M")

In [ ]:
ecdet_l_config = {
    "backbone": {
        "name": "ecvits",
        "skip_load_backbone": True,
        "interaction_indexes": [10, 11],
        "embed_dim": 384,
        "num_heads": 6,
        "patch_size": 16,
        "proj_dim": 256,
        "num_levels": 3,
        "embed_layer": "ConvPyramidPatchEmbed",
        "ffn_layer": "mlp",
        "ffn_ratio": 4,
        "skip_load_backbone": False,
    },
    "encoder": {
        "in_channels": [256, 256, 256],
        "feat_strides": [8, 16, 32],
        "hidden_dim": 256,
        "nhead": 8,
        "dim_feedforward": 1024,
        "dropout": 0.0,
        "use_encoder_idx": [2],
        "num_encoder_layers": 1,
        "pe_temperature": 10000,
        "expansion": 0.75,
        "depth_mult": 1,
        "act": "silu",
        "eval_spatial_size": None,
        "csp_type": "csp2",
        "fuse_op": "sum",
    },
    "decoder": {
        "num_classes": 80,
        "hidden_dim": 256,
        "num_queries": 300,
        "feat_channels": [256, 256, 256],
        "feat_strides": [8, 16, 32],
        "num_levels": 3,
        "num_points": [3, 6, 3],
        "nhead": 8,
        "num_layers": 4,
        "dim_feedforward": 1024,
        "dropout": 0.0,
        "activation": "silu",
        "num_denoising": 100,
        "label_noise_ratio": 0.5,
        "box_noise_scale": 1.0,
        "learn_query_content": False,
        "eval_spatial_size": [640, 640],
        "eval_idx": -1,
        "eps": 1e-2,
        "aux_loss": True,
        "cross_attn_method": "default",
        "query_select_method": "default",
        "reg_max": 32,
        "reg_scale": 4,
        "layer_scale": 1,
        "share_bbox_head": False,
        "share_score_head": False,
        "mask_downsample_ratio": None,
    },
    "postprocessor": {
        "num_classes": 80,
        "use_focal_loss": True,
        "num_top_queries": 300,
        "remap_mscoco_category": False,
    },
}

ecdet_l = ECDet(ecdet_l_config).from_pretrained("Intellindust/ECDet_L")

In [ ]:
ecdet_x_config = {
    "backbone": {
        "name": "ecvitsplus",
        "skip_load_backbone": True,
        "interaction_indexes": [10, 11],
        "embed_dim": 384,
        "num_heads": 6,
        "patch_size": 16,
        "proj_dim": 256,
        "num_levels": 3,
        "embed_layer": "ConvPyramidPatchEmbed",
        "ffn_layer": "mlp",
        "ffn_ratio": 6,
        "skip_load_backbone": False,
    },
    "encoder": {
        "in_channels": [256, 256, 256],
        "feat_strides": [8, 16, 32],
        "hidden_dim": 256,
        "nhead": 8,
        "dim_feedforward": 2048,
        "dropout": 0.0,
        "use_encoder_idx": [2],
        "num_encoder_layers": 1,
        "pe_temperature": 10000,
        "expansion": 1.5,
        "depth_mult": 1,
        "act": "silu",
        "eval_spatial_size": None,
        "csp_type": "csp2",
        "fuse_op": "sum",
    },
    "decoder": {
        "num_classes": 80,
        "hidden_dim": 256,
        "num_queries": 300,
        "feat_channels": [256, 256, 256],
        "feat_strides": [8, 16, 32],
        "num_levels": 3,
        "num_points": [3, 6, 3],
        "nhead": 8,
        "num_layers": 4,
        "dim_feedforward": 2048,
        "dropout": 0.0,
        "activation": "silu",
        "num_denoising": 100,
        "label_noise_ratio": 0.5,
        "box_noise_scale": 1.0,
        "learn_query_content": False,
        "eval_spatial_size": [640, 640],
        "eval_idx": -1,
        "eps": 1e-2,
        "aux_loss": True,
        "cross_attn_method": "default",
        "query_select_method": "default",
        "reg_max": 32,
        "reg_scale": 4,
        "layer_scale": 1,
        "share_bbox_head": False,
        "share_score_head": False,
        "mask_downsample_ratio": None,
    },
    "postprocessor": {
        "num_classes": 80,
        "use_focal_loss": True,
        "num_top_queries": 300,
        "remap_mscoco_category": False,
    },
}

ecdet_x = ECDet(ecdet_x_config).from_pretrained("Intellindust/ECDet_X")

# ECSeg

In [ ]:
import torch
import torch.nn as nn
from huggingface_hub import PyTorchModelHubMixin

from ecdetseg.engine.edgecrafter.ecvit import ViTAdapter
from ecdetseg.engine.edgecrafter.hybrid_encoder import HybridEncoder
from ecdetseg.engine.edgecrafter.decoder import ECTransformer
from ecdetseg.engine.edgecrafter.postprocessor import PostProcessor


class ECSeg(nn.Module, PyTorchModelHubMixin):
    def __init__(self, config):
        super().__init__()
        self.config = config

        self.backbone = ViTAdapter(
            **config["backbone"]
        )
        self.encoder = HybridEncoder(
            **config["encoder"]
        )
        self.decoder = ECTransformer(
            **config["decoder"]
        )
        self.postprocessor = PostProcessor(
            **config["postprocessor"]
        )
    
    def forward(self, x, orig_target_sizes):
        x = self.backbone(x)
        x = self.encoder(x)
        x = self.decoder(x)
        x = self.postprocessor(x, orig_target_sizes)

        return x

In [ ]:
ecseg_s_config = {
    "backbone": {
        "name": "ecseg_vitt",
        "skip_load_backbone": True,
        "interaction_indexes": [10, 11],
        "embed_dim": 192,
        "num_heads": 3,
        "patch_size": 16,
        "proj_dim": None,
        "num_levels": 3,
        "embed_layer": "ConvPyramidPatchEmbed",
        "ffn_layer": "mlp",
        "ffn_ratio": 4,
        "skip_load_backbone": False,
    },
    "encoder": {
        "in_channels": [192, 192, 192],
        "feat_strides": [8, 16, 32],
        "hidden_dim": 192,
        "nhead": 8,
        "dim_feedforward": 512,
        "dropout": 0.0,
        "use_encoder_idx": [2],
        "num_encoder_layers": 1,
        "pe_temperature": 10000,
        "expansion": 0.34,
        "depth_mult": 0.67,
        "act": "silu",
        "eval_spatial_size": None,
        "csp_type": "csp2",
        "fuse_op": "sum",
    },
    "decoder": {
        "num_classes": 80,
        "hidden_dim": 192,
        "num_queries": 300,
        "feat_channels": [192, 192, 192],
        "feat_strides": [8, 16, 32],
        "num_levels": 3,
        "num_points": [3, 6, 3],
        "nhead": 8,
        "num_layers": 4,
        "dim_feedforward": 512,
        "dropout": 0.0,
        "activation": "silu",
        "num_denoising": 100,
        "label_noise_ratio": 0.5,
        "box_noise_scale": 1.0,
        "learn_query_content": False,
        "eval_spatial_size": [640, 640],
        "eval_idx": -1,
        "eps": 1e-2,
        "aux_loss": True,
        "cross_attn_method": "default",
        "query_select_method": "default",
        "reg_max": 32,
        "reg_scale": 4,
        "layer_scale": 1,
        "share_bbox_head": False,
        "share_score_head": False,
        "mask_downsample_ratio": 4,
    },
    "postprocessor": {
        "num_classes": 80,
        "use_focal_loss": True,
        "num_top_queries": 300,
        "remap_mscoco_category": False,
    },
}

ecseg_s = ECSeg(ecseg_s_config).from_pretrained("Intellindust/ECSeg_S")

In [ ]:
ecseg_m_config = {
    "backbone": {
        "name": "ecseg_vittplus",
        "skip_load_backbone": True,
        "interaction_indexes": [10, 11],
        "embed_dim": 256,
        "num_heads": 4,
        "patch_size": 16,
        "proj_dim": None,
        "num_levels": 3,
        "embed_layer": "ConvPyramidPatchEmbed",
        "ffn_layer": "mlp",
        "ffn_ratio": 4,
        "skip_load_backbone": False,
    },
    "encoder": {
        "in_channels": [256, 256, 256],
        "feat_strides": [8, 16, 32],
        "hidden_dim": 256,
        "nhead": 8,
        "dim_feedforward": 512,
        "dropout": 0.0,
        "use_encoder_idx": [2],
        "num_encoder_layers": 1,
        "pe_temperature": 10000,
        "expansion": 0.75,
        "depth_mult": 0.67,
        "act": "silu",
        "eval_spatial_size": None,
        "csp_type": "csp2",
        "fuse_op": "sum",
    },
    "decoder": {
        "num_classes": 80,
        "hidden_dim": 256,
        "num_queries": 300,
        "feat_channels": [256, 256, 256],
        "feat_strides": [8, 16, 32],
        "num_levels": 3,
        "num_points": [3, 6, 3],
        "nhead": 8,
        "num_layers": 4,
        "dim_feedforward": 1024,
        "dropout": 0.0,
        "activation": "silu",
        "num_denoising": 100,
        "label_noise_ratio": 0.5,
        "box_noise_scale": 1.0,
        "learn_query_content": False,
        "eval_spatial_size": [640, 640],
        "eval_idx": -1,
        "eps": 1e-2,
        "aux_loss": True,
        "cross_attn_method": "default",
        "query_select_method": "default",
        "reg_max": 32,
        "reg_scale": 4,
        "layer_scale": 1,
        "share_bbox_head": False,
        "share_score_head": False,
        "mask_downsample_ratio": 4,
    },
    "postprocessor": {
        "num_classes": 80,
        "use_focal_loss": True,
        "num_top_queries": 300,
        "remap_mscoco_category": False,
    },
}

ecseg_m = ECSeg(ecseg_m_config).from_pretrained("Intellindust/ECSeg_M")

In [ ]:
ecseg_l_config = {
    "backbone": {
        "name": "ecseg_vits",
        "skip_load_backbone": True,
        "interaction_indexes": [10, 11],
        "embed_dim": 384,
        "num_heads": 6,
        "patch_size": 16,
        "proj_dim": 256,
        "num_levels": 3,
        "embed_layer": "ConvPyramidPatchEmbed",
        "ffn_layer": "mlp",
        "ffn_ratio": 4,
        "skip_load_backbone": False,
    },
    "encoder": {
        "in_channels": [256, 256, 256],
        "feat_strides": [8, 16, 32],
        "hidden_dim": 256,
        "nhead": 8,
        "dim_feedforward": 1024,
        "dropout": 0.0,
        "use_encoder_idx": [2],
        "num_encoder_layers": 1,
        "pe_temperature": 10000,
        "expansion": 0.75,
        "depth_mult": 1,
        "act": "silu",
        "eval_spatial_size": None,
        "csp_type": "csp2",
        "fuse_op": "sum",
    },
    "decoder": {
        "num_classes": 80,
        "hidden_dim": 256,
        "num_queries": 300,
        "feat_channels": [256, 256, 256],
        "feat_strides": [8, 16, 32],
        "num_levels": 3,
        "num_points": [3, 6, 3],
        "nhead": 8,
        "num_layers": 4,
        "dim_feedforward": 1024,
        "dropout": 0.0,
        "activation": "silu",
        "num_denoising": 100,
        "label_noise_ratio": 0.5,
        "box_noise_scale": 1.0,
        "learn_query_content": False,
        "eval_spatial_size": [640, 640],
        "eval_idx": -1,
        "eps": 1e-2,
        "aux_loss": True,
        "cross_attn_method": "default",
        "query_select_method": "default",
        "reg_max": 32,
        "reg_scale": 4,
        "layer_scale": 1,
        "share_bbox_head": False,
        "share_score_head": False,
        "mask_downsample_ratio": 4,
    },
    "postprocessor": {
        "num_classes": 80,
        "use_focal_loss": True,
        "num_top_queries": 300,
        "remap_mscoco_category": False,
    },
}

ecseg_l = ECSeg(ecseg_l_config).from_pretrained("Intellindust/ECSeg_L")

In [ ]:
ecseg_x_config = {
    "backbone": {
        "name": "ecseg_vitsplus",
        "skip_load_backbone": True,
        "interaction_indexes": [10, 11],
        "embed_dim": 384,
        "num_heads": 6,
        "patch_size": 16,
        "proj_dim": 256,
        "num_levels": 3,
        "embed_layer": "ConvPyramidPatchEmbed",
        "ffn_layer": "mlp",
        "ffn_ratio": 6,
        "skip_load_backbone": False,
    },
    "encoder": {
        "in_channels": [256, 256, 256],
        "feat_strides": [8, 16, 32],
        "hidden_dim": 256,
        "nhead": 8,
        "dim_feedforward": 2048,
        "dropout": 0.0,
        "use_encoder_idx": [2],
        "num_encoder_layers": 1,
        "pe_temperature": 10000,
        "expansion": 1.5,
        "depth_mult": 1,
        "act": "silu",
        "eval_spatial_size": None,
        "csp_type": "csp2",
        "fuse_op": "sum",
    },
    "decoder": {
        "num_classes": 80,
        "hidden_dim": 256,
        "num_queries": 300,
        "feat_channels": [256, 256, 256],
        "feat_strides": [8, 16, 32],
        "num_levels": 3,
        "num_points": [3, 6, 3],
        "nhead": 8,
        "num_layers": 4,
        "dim_feedforward": 2048,
        "dropout": 0.0,
        "activation": "silu",
        "num_denoising": 100,
        "label_noise_ratio": 0.5,
        "box_noise_scale": 1.0,
        "learn_query_content": False,
        "eval_spatial_size": [640, 640],
        "eval_idx": -1,
        "eps": 1e-2,
        "aux_loss": True,
        "cross_attn_method": "default",
        "query_select_method": "default",
        "reg_max": 32,
        "reg_scale": 4,
        "layer_scale": 1,
        "share_bbox_head": False,
        "share_score_head": False,
        "mask_downsample_ratio": 4,
    },
    "postprocessor": {
        "num_classes": 80,
        "use_focal_loss": True,
        "num_top_queries": 300,
        "remap_mscoco_category": False,
    },
}

ecseg_x = ECSeg(ecseg_x_config).from_pretrained("Intellindust/ECSeg_X")

# ECPose

In [ ]:
import torch
import torch.nn as nn
from huggingface_hub import PyTorchModelHubMixin

from ecpose.engine.edgecrafter.ecvit import ViTAdapter
from ecpose.engine.edgecrafter.hybrid_encoder import HybridEncoder
from ecpose.engine.edgecrafter.detrpose_transformer import DETRTransformer
from ecpose.engine.edgecrafter.detrpose_postprocesses import DETRPosePostProcessor


class ECPose(nn.Module, PyTorchModelHubMixin):
    def __init__(self, config):
        super().__init__()
        self.config = config

        self.backbone = ViTAdapter(
            **config["backbone"]
        )
        self.encoder = HybridEncoder(
            **config["encoder"]
        )
        self.decoder = DETRTransformer(
            **config["decoder"]
        )
        self.postprocessor = DETRPosePostProcessor(
            **config["postprocessor"]
        )
    
    def forward(self, x, orig_target_sizes):
        x = self.backbone(x)
        x = self.encoder(x)
        x = self.decoder(x)
        x = self.postprocessor(x, orig_target_sizes)

        return x

In [ ]:
ecpose_s_config = {
    "backbone": {
        "name": "ecpose_vitt",
        "skip_load_backbone": True,
        "interaction_indexes": [10, 11],
        "embed_dim": 192,
        "num_heads": 3,
        "patch_size": 16,
        "proj_dim": None,
        "num_levels": 3,
        "embed_layer": "ConvPyramidPatchEmbed",
        "ffn_layer": "mlp",
        "ffn_ratio": 4,
        "skip_load_backbone": False,
    },
    "encoder": {
        "in_channels": [192, 192, 192],
        "feat_strides": [8, 16, 32],
        "hidden_dim": 192,
        "use_encoder_idx": [2],
        "num_encoder_layers": 1,
        "nhead": 8,
        "dim_feedforward": 512,
        "dropout": 0.0,
        "expansion": 0.34,
        "depth_mult": 0.67,
        "act": "silu",
        "csp_type": "csp2",
        "fuse_op": "sum",
    },
    "decoder": {
        "hidden_dim": 192,
        "dropout": 0.0,
        "nhead": 8,
        "num_queries": 60,
        "dim_feedforward": 512,
        "num_decoder_layers": 3,
        "normalize_before": False,
        "return_intermediate_dec": True,
        "activation": "relu",
        "num_feature_levels": 3,
        "dec_n_points": 4,
        "learnable_tgt_init": True,
        "two_stage_type": "standard",
        "num_body_points": 17,
        "aux_loss": True,
        "dec_pred_class_embed_share": False,
        "dec_pred_pose_embed_share": False,
        "two_stage_class_embed_share": False,
        "two_stage_bbox_embed_share": False,
        "cls_no_bias": False,
        "feat_strides": [8, 16, 32],
        "reg_max": 32,
        "reg_scale": 4,
        "eval_spatial_size": [640, 640],
    },
    "postprocessor": {
        "num_select": 60,
        "num_body_points": 17,
    },
}

ecpose_s = ECPose(ecpose_s_config).from_pretrained("Intellindust/ECPose_S")

In [ ]:
ecpose_m_config = {
    "backbone": {
        "name": "ecpose_vittplus",
        "skip_load_backbone": True,
        "interaction_indexes": [10, 11],
        "embed_dim": 256,
        "num_heads": 4,
        "patch_size": 16,
        "proj_dim": None,
        "num_levels": 3,
        "embed_layer": "ConvPyramidPatchEmbed",
        "ffn_layer": "mlp",
        "ffn_ratio": 4,
        "skip_load_backbone": False,
    },
    "encoder": {
        "in_channels": [256, 256, 256],
        "feat_strides": [8, 16, 32],
        "hidden_dim": 256,
        "use_encoder_idx": [2],
        "num_encoder_layers": 1,
        "nhead": 8,
        "dim_feedforward": 512,
        "dropout": 0.0,
        "expansion": 0.75,
        "depth_mult": 0.67,
        "act": "silu",
        "csp_type": "csp2",
        "fuse_op": "sum",
    },
    "decoder": {
        "hidden_dim": 256,
        "dropout": 0.0,
        "nhead": 8,
        "num_queries": 60,
        "dim_feedforward": 512,
        "num_decoder_layers": 4,
        "normalize_before": False,
        "return_intermediate_dec": True,
        "activation": "relu",
        "num_feature_levels": 3,
        "dec_n_points": 4,
        "learnable_tgt_init": True,
        "two_stage_type": "standard",
        "num_body_points": 17,
        "aux_loss": True,
        "dec_pred_class_embed_share": False,
        "dec_pred_pose_embed_share": False,
        "two_stage_class_embed_share": False,
        "two_stage_bbox_embed_share": False,
        "cls_no_bias": False,
        "feat_strides": [8, 16, 32],
        "reg_max": 32,
        "reg_scale": 4,
        "eval_spatial_size": [640, 640],
    },
    "postprocessor": {
        "num_select": 60,
        "num_body_points": 17,
    },
}

ecpose_m = ECPose(ecpose_m_config).from_pretrained("Intellindust/ECPose_M")

In [ ]:
ecpose_l_config = {
    "backbone": {
        "name": "ecpose_vits",
        "skip_load_backbone": True,
        "interaction_indexes": [10, 11],
        "embed_dim": 384,
        "num_heads": 6,
        "patch_size": 16,
        "proj_dim": 256,
        "num_levels": 3,
        "embed_layer": "ConvPyramidPatchEmbed",
        "ffn_layer": "mlp",
        "ffn_ratio": 4,
        "skip_load_backbone": False,
    },
    "encoder": {
        "in_channels": [256, 256, 256],
        "feat_strides": [8, 16, 32],
        "hidden_dim": 256,
        "use_encoder_idx": [2],
        "num_encoder_layers": 1,
        "nhead": 8,
        "dim_feedforward": 1024,
        "dropout": 0.0,
        "expansion": 0.75,
        "depth_mult": 1,
        "act": "silu",
        "csp_type": "csp2",
        "fuse_op": "sum",
    },
    "decoder": {
        "hidden_dim": 256,
        "dropout": 0.0,
        "nhead": 8,
        "num_queries": 60,
        "dim_feedforward": 1024,
        "num_decoder_layers": 4,
        "normalize_before": False,
        "return_intermediate_dec": True,
        "activation": "relu",
        "num_feature_levels": 3,
        "dec_n_points": 4,
        "learnable_tgt_init": True,
        "two_stage_type": "standard",
        "num_body_points": 17,
        "aux_loss": True,
        "dec_pred_class_embed_share": False,
        "dec_pred_pose_embed_share": False,
        "two_stage_class_embed_share": False,
        "two_stage_bbox_embed_share": False,
        "cls_no_bias": False,
        "feat_strides": [8, 16, 32],
        "reg_max": 32,
        "reg_scale": 4,
        "eval_spatial_size": [640, 640],
    },
    "postprocessor": {
        "num_select": 60,
        "num_body_points": 17,
    },
}

ecpose_l = ECPose(ecpose_l_config).from_pretrained("Intellindust/ECPose_L")

In [ ]:
ecpose_x_config = {
    "backbone": {
        "name": "ecpose_vitsplus",
        "skip_load_backbone": True,
        "interaction_indexes": [10, 11],
        "embed_dim": 384,
        "num_heads": 6,
        "patch_size": 16,
        "proj_dim": 256,
        "num_levels": 3,
        "embed_layer": "ConvPyramidPatchEmbed",
        "ffn_layer": "mlp",
        "ffn_ratio": 6,
        "skip_load_backbone": False,
    },
    "encoder": {
        "in_channels": [256, 256, 256],
        "feat_strides": [8, 16, 32],
        "hidden_dim": 256,
        "use_encoder_idx": [2],
        "num_encoder_layers": 1,
        "nhead": 8,
        "dim_feedforward": 2048,
        "dropout": 0.0,
        "expansion": 1.5,
        "depth_mult": 1,
        "act": "silu",
        "csp_type": "csp2",
        "fuse_op": "sum",
    },
    "decoder": {
        "hidden_dim": 256,
        "dropout": 0.0,
        "nhead": 8,
        "num_queries": 60,
        "dim_feedforward": 2048,
        "num_decoder_layers": 4,
        "normalize_before": False,
        "return_intermediate_dec": True,
        "activation": "relu",
        "num_feature_levels": 3,
        "dec_n_points": 4,
        "learnable_tgt_init": True,
        "two_stage_type": "standard",
        "num_body_points": 17,
        "aux_loss": True,
        "dec_pred_class_embed_share": False,
        "dec_pred_pose_embed_share": False,
        "two_stage_class_embed_share": False,
        "two_stage_bbox_embed_share": False,
        "cls_no_bias": False,
        "feat_strides": [8, 16, 32],
        "reg_max": 32,
        "reg_scale": 4,
        "eval_spatial_size": [640, 640],
    },
    "postprocessor": {
        "num_select": 60,
        "num_body_points": 17,
    },
}

ecpose_x = ECPose(ecpose_x_config).from_pretrained("Intellindust/ECPose_X")